# Week 1 Exercise: Technical Tutor

A tool that takes a technical question and explains it, answered by Gemini (streamed)
and by Llama 3.2 running locally through Ollama. Both go through the same `openai` client library.

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
MODEL_GEMINI = "gemini-3.1-flash-lite"
MODEL_LLAMA = "llama3.2"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No API key was found. Add GEMINI_API_KEY to your .env file.")
elif not api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
elif api_key.strip() != api_key:
    print("The API key has leading or trailing whitespace.")
else:
    print("API key found and looks good so far!")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
question = """
What is RAG (retrieval-augmented generation) at a high level, and why does it exist?
"""

system_prompt = (
    "You are a helpful technical tutor who answers questions about python code, "
    "software engineering, data science and LLMs. Explain clearly, with structure, "
    "and respond in markdown without code fences around the whole answer."
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Please give a detailed explanation to the following question: " + question},
]

## Gemini, streamed

In [ ]:
stream = gemini.chat.completions.create(model=MODEL_GEMINI, messages=messages, stream=True)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

## Llama 3.2, local

In [ ]:
response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)
display(Markdown(response.choices[0].message.content))

## Try another question

Change `question`, re-run from that cell, and compare the two answers.
On a slower machine, use `MODEL_LLAMA = "llama3.2:1b"`.